# Kuroshio manuscript figures 2–8 | unified-font redraw notebook (WSL)

这个 notebook 的目标是：

1. **不要再直接拉伸图片改字体**；
2. 回到原始/中间结果表，**重新绘制 Figure 2–8**；
3. 全部图统一使用同一套字体、字号、线宽和导出参数；
4. 适合在 **Windows + WSL** 环境直接运行。

## 说明

- 这个 notebook 主要整合了你目前上传文件中能够定位到的绘图逻辑：
  - Figure 2：`rewarite1_WSL(4).ipynb`
  - Figure 3 / Figure 4：`Kuroshio_R79_ADT_GLORYS_Smoothed_Dt_3Panels_WSL(1).ipynb` 的输出表
  - Figure 5：`Kuroshio_Daily_R4_MultiYear_Rolling_Hindcast_RTX5080_WSL(2).ipynb` 的输出表
  - Figure 6：从 R4 / R3 的误差汇总表中自动构造一个统一风格图
  - Figure 7：需要 **section / transect / lead / model / rmse** 这类长表；会自动搜索，找不到时会提示你补路径
  - Figure 8：需要 **noise sensitivity** 和 **seed stability** 两个表；会自动搜索，找不到时会提示你补路径

## 你需要做的事

最重要的是先检查 **CONFIG** 单元：

- `PROJECT_ROOT`
- `OUTPUT_DIR`
- `FIG5_MODEL_NAME`（如果不想自动识别主模型）
- `FIG7_TABLE`, `FIG8_NOISE_TABLE`, `FIG8_SEED_TABLE`（如果自动搜索失败，就手动指定）

运行顺序建议：

1. 依次运行前面的配置与函数单元；
2. 先运行 Figure 2–6；
3. 再运行 Figure 7 / 8；
4. 最后运行“Run all available figures”单元。


In [ ]:
# ==== 0. Imports and global configuration ====
from __future__ import annotations

import os
import re
import json
import math
import glob
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d
from scipy.stats import gaussian_kde

# ---------------------- USER CONFIG ----------------------
# 改成你在 WSL 中的工程根目录。
PROJECT_ROOT = Path(os.environ.get('KUROSHIO_ROOT', '/mnt/d/KuroshioDaily1993_2025')).expanduser()

# 输出目录
OUTPUT_DIR = PROJECT_ROOT / 'MANUSCRIPT_FIGURES_UNIFIED_FONTS'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Figure 2 日流轴时间范围
FIG2_START = '1993-01-01'
FIG2_END   = '2024-12-31'
FIG2_DAILY_STRIDE = 1
FIG2_SMOOTH_KM = 10.0
FIG2_MAX_LINES = 0  # 0 = all

# Figure 5 指定主模型名；None 表示自动识别
FIG5_MODEL_NAME = None

# 如果自动搜索失败，可在这里手动指定 CSV 路径
FIG3_DAILY_TABLE = None   # e.g. PROJECT_ROOT/'R79.../tables/01_smoothed_daily_path_distance.csv'
FIG3_SECTION_TABLE = None # e.g. PROJECT_ROOT/'R79.../tables/02_smoothed_daily_section_intersections.csv'
FIG5_TABLE = None         # e.g. PROJECT_ROOT/'R4.../data/08_逐年模型汇总.csv'
FIG6_TABLE = None         # 可选；若为空则自动从 R4 / R3 搜索
FIG7_TABLE = None         # section/transect long-format RMSE table
FIG8_NOISE_TABLE = None   # noise sensitivity table
FIG8_SEED_TABLE = None    # seed stability table

# Figure 7 / 8 自动搜索失败时，可改这些模式
FIG7_GLOB_PATTERNS = [
    '**/*section*rmse*.csv', '**/*transect*rmse*.csv',
    '**/*section*performance*.csv', '**/*transect*performance*.csv',
    '**/*断面*RMSE*.csv', '**/*断面*性能*.csv'
]
FIG8_NOISE_GLOB_PATTERNS = [
    '**/*noise*sensitivity*.csv', '**/*input*noise*.csv', '**/*噪声*敏感*.csv'
]
FIG8_SEED_GLOB_PATTERNS = [
    '**/*seed*stability*.csv', '**/*random*seed*.csv', '**/*种子*稳定*.csv'
]

# 统一样式
FONT_FAMILY = 'Times New Roman'   # 如果系统没有，会自动 fallback
TITLE_SIZE = 18
SUBTITLE_SIZE = 16
LABEL_SIZE = 16
TICK_SIZE = 14
LEGEND_SIZE = 13
LINE_WIDTH = 2.0
AXES_LINE_WIDTH = 1.2
FIG_DPI = 600

# 配色
COLORS = {
    'blue': '#2F7EAE',
    'orange': '#D07A31',
    'green': '#4A9C78',
    'gray': '#7E8790',
    'dark': '#404040',
    'transect': '#BC4C51',
    'meanaxis': '#249267',
    'fill_blue': '#BFD7E3',
}

SECTIONS = {
    'A': ((121.50, 23.50), (125.00, 22.75)),
    'B': ((121.50, 26.25), (124.25, 24.25)),
    'PN': ((126.00, 29.00), (128.15, 27.30)),
    'C': ((130.25, 31.25), (132.50, 27.00)),
    'D': ((137.00, 34.75), (138.00, 27.75)),
    'E': ((141.75, 40.00), (145.00, 31.50)),
}
SECTION_ORDER = ['A','B','PN','C','D','E']
LEADS = [30, 60, 90]


In [ ]:
# ==== 1. Unified style helpers ====
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'font.family': FONT_FAMILY,
    'font.size': TICK_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'axes.titleweight': 'bold',
    'axes.labelsize': LABEL_SIZE,
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'axes.linewidth': AXES_LINE_WIDTH,
    'grid.alpha': 0.18,
    'grid.linewidth': 0.6,
    'grid.linestyle': '--',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.unicode_minus': False,
})


def finish_axis(ax, xlabel=None, ylabel=None, title=None):
    if title is not None:
        ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold', pad=10)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=LABEL_SIZE)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=1.0)
    ax.grid(True, alpha=0.18)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(AXES_LINE_WIDTH)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)


def save_figure(fig, stem: str):
    png = OUTPUT_DIR / f'{stem}.png'
    pdf = OUTPUT_DIR / f'{stem}.pdf'
    fig.tight_layout()
    fig.savefig(png, dpi=FIG_DPI, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    print(f'[saved] {png}')
    return png, pdf


def find_first(root: Path, patterns, must_exist=True):
    if root is None:
        return None
    for pat in patterns:
        matches = sorted(root.glob(pat))
        if matches:
            return matches[0]
    if must_exist:
        raise FileNotFoundError(f'No file matched patterns: {patterns}')
    return None


def read_csv_flex(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    # 尝试自动识别编码
    for enc in ['utf-8', 'utf-8-sig', 'gbk', 'gb18030']:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)


def choose_column(df, candidates, required=True):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in cols:
            return cols[cand.lower()]
    if required:
        raise KeyError(f'Cannot find any of columns {candidates}; available={list(df.columns)}')
    return None


print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR   =', OUTPUT_DIR)


In [ ]:
# ==== 2. Figure 2 helpers: daily GLORYS axes and six transects ====
def fig2_load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        lon_key = 'lon' if 'lon' in z.files else 'axis_lon'
        lat_key = 'lat' if 'lat' in z.files else 'axis_lat'
        x = np.column_stack([np.asarray(z[lon_key]).reshape(-1), np.asarray(z[lat_key]).reshape(-1)])
    x = x[np.isfinite(x).all(axis=1)]
    if len(x) < 10:
        raise ValueError(f'{path} has too few valid points')
    return x


def fig2_simple_smooth_path(path, sigma_km=10, max_displace_km=12):
    xy = np.asarray(path, float)
    if len(xy) < 10:
        return xy
    lat_mid = np.deg2rad((xy[:-1,1] + xy[1:,1]) / 2)
    d = np.sqrt((111.32*np.cos(lat_mid)*np.diff(xy[:,0]))**2 + (111.32*np.diff(xy[:,1]))**2)
    xdist = np.r_[0, np.cumsum(d)]
    good = np.r_[True, d > 0.03]
    if good.sum() < 10 or xdist[-1] < 30:
        return xy
    s = np.arange(0, xdist[-1], 5)
    if len(s) < 10:
        return xy
    if s[-1] != xdist[-1]:
        s = np.r_[s, xdist[-1]]
    x = np.interp(s, xdist[good], xy[good,0])
    y = np.interp(s, xdist[good], xy[good,1])
    xyi = np.column_stack([x,y])
    smooth = gaussian_filter1d(xyi, sigma=max(.01, sigma_km/5), axis=0, mode='nearest')
    smooth[[0,-1]] = xyi[[0,-1]]
    dx = (smooth[:,0]-xyi[:,0]) * 111.32 * np.cos(np.deg2rad(xyi[:,1]))
    dy = (smooth[:,1]-xyi[:,1]) * 111.32
    delta = np.sqrt(dx*dx + dy*dy)
    fac = np.minimum(1, max_displace_km/np.maximum(delta, .0001))
    res = xyi + (smooth-xyi) * fac[:,None]
    res[[0,-1]] = xyi[[0,-1]]
    return res


def try_draw_coast(ax):
    # 1) optional cartopy coastlines
    try:
        import cartopy.crs as ccrs  # noqa
        import cartopy.feature as cfeature
        # 如果当前 ax 不是 geo axes，就直接跳过 cartopy
        return False
    except Exception:
        pass
    # 2) fallback: no-op
    return False


def plot_figure2():
    track_dir = PROJECT_ROOT / 'final' / 'tracks_final'
    if not track_dir.is_dir():
        raise FileNotFoundError(f'Figure 2 requires daily GLORYS NPZ under: {track_dir}')

    files = []
    for f in track_dir.glob('????????.npz'):
        if f.stem.isdigit():
            day = pd.to_datetime(f.stem, format='%Y%m%d', errors='coerce')
            if pd.notna(day) and pd.Timestamp(FIG2_START) <= day <= pd.Timestamp(FIG2_END):
                files.append((day, f))
    files.sort(key=lambda x: x[0])
    files = files[::max(1, FIG2_DAILY_STRIDE)]
    if FIG2_MAX_LINES > 0:
        files = files[:FIG2_MAX_LINES]
    if not files:
        raise RuntimeError('No daily axis files found in requested time range')

    paths = []
    profiles = []
    for day, f in files:
        try:
            x = fig2_load_npz(f)
            x = fig2_simple_smooth_path(x, FIG2_SMOOTH_KM)
            paths.append(x)
            d = np.sqrt((111.32*np.cos(np.deg2rad((x[1:,1]+x[:-1,1])/2))*np.diff(x[:,0]))**2 +
                        (111.32*np.diff(x[:,1]))**2)
            s = np.r_[0, np.cumsum(d)]
            if s[-1] > 100 and np.all(np.diff(s) > 0):
                q = np.linspace(0, s[-1], 220)
                profiles.append(np.column_stack([np.interp(q, s, x[:,0]), np.interp(q, s, x[:,1])]))
        except Exception as e:
            print('[Figure 2 skip]', f.name, e)

    fig, ax = plt.subplots(figsize=(12.2, 9.4))
    ax.set_facecolor('white')
    _ = try_draw_coast(ax)

    collection = LineCollection(paths, colors=COLORS['gray'], linewidths=.30, alpha=.04, zorder=2, rasterized=True)
    ax.add_collection(collection)

    if profiles:
        pr = np.stack(profiles)
        mu = np.nanmean(pr, axis=0)
        mu = fig2_simple_smooth_path(mu, 18, 12)
        ax.plot(mu[:,0], mu[:,1], color=COLORS['meanaxis'], lw=3.0, zorder=4)

    for name, (p0, p1) in SECTIONS.items():
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], lw=2.2, color=COLORS['transect'], zorder=5)
        frac = 0.67
        where = np.array(p0)*(1-frac) + np.array(p1)*frac
        offsets = {'A':(.25,-.5),'B':(-.6,.2),'PN':(.25,.45),'C':(.3,.45),'D':(.5,-.1),'E':(.45,-.5)}
        dx, dy = offsets[name]
        ax.text(where[0]+dx, where[1]+dy, name, color='#B23841', fontsize=SUBTITLE_SIZE+2, weight='semibold', zorder=7)

    ax.set_xlim(119, 147)
    ax.set_ylim(20, 42)
    ax.set_xticks(np.arange(120, 148, 4))
    ax.set_yticks(np.arange(20, 44, 4))
    ax.set_xticklabels([f'{i}°E' for i in np.arange(120, 148, 4)], fontsize=TICK_SIZE)
    ax.set_yticklabels([f'{i}°N' for i in np.arange(20, 44, 4)], fontsize=TICK_SIZE)
    ax.set_aspect(1/np.cos(np.deg2rad(31)), adjustable='box')
    ax.grid(lw=.6, color='#AAB3B8', alpha=.26)
    finish_axis(ax, xlabel='Longitude', ylabel='Latitude', title='Daily identified Kuroshio axes and six diagnostic transects')

    ax.legend(handles=[
        Line2D([0],[0], color=COLORS['gray'], lw=2, label='Daily GLORYS 50 m axis'),
        Line2D([0],[0], color=COLORS['meanaxis'], lw=3, label='Mean axis'),
        Line2D([0],[0], color=COLORS['transect'], lw=2, label='Six transects')
    ], frameon=False, fontsize=LEGEND_SIZE, loc='lower right')

    return save_figure(fig, 'Figure2_daily_GLORYS_axis_6_transects_unified')


In [ ]:
# ==== 3. Figure 3 / 4 data loaders (from R79 outputs) ====
def locate_r79_tables():
    daily_path = Path(FIG3_DAILY_TABLE) if FIG3_DAILY_TABLE else None
    sec_path = Path(FIG3_SECTION_TABLE) if FIG3_SECTION_TABLE else None

    if daily_path is None or sec_path is None:
        # 优先搜索 R79 结果目录
        patterns_daily = [
            '**/R79*/tables/01_smoothed_daily_path_distance.csv',
            '**/01_smoothed_daily_path_distance.csv',
        ]
        patterns_sec = [
            '**/R79*/tables/02_smoothed_daily_section_intersections.csv',
            '**/02_smoothed_daily_section_intersections.csv',
        ]
        if daily_path is None:
            daily_path = find_first(PROJECT_ROOT, patterns_daily)
        if sec_path is None:
            sec_path = find_first(PROJECT_ROOT, patterns_sec)
    print('[Figure3/4] daily table  =', daily_path)
    print('[Figure3/4] section table=', sec_path)
    return daily_path, sec_path


def load_r79_tables():
    daily_path, sec_path = locate_r79_tables()
    daily = read_csv_flex(daily_path)
    sec = read_csv_flex(sec_path)
    daily['date'] = pd.to_datetime(daily['date'])
    sec['date'] = pd.to_datetime(sec['date'])
    return daily, sec


def yearly_summary_from_daily(daily):
    g = daily.copy()
    g['year'] = pd.to_datetime(g['date']).dt.year
    out = g.groupby('year', as_index=False).agg(
        D_mean_km=('D_mean_km', 'mean'),
        P25=('D_mean_km', lambda x: np.nanquantile(x, .25)),
        P75=('D_mean_km', lambda x: np.nanquantile(x, .75)),
        n_valid=('D_mean_km', 'count')
    )
    out['sufficient'] = out['n_valid'] >= 180
    return out


def monthly_section_anomalies(sec):
    use = sec.copy()
    use = use[use['status'].astype(str).str.contains('unique', case=False, na=False)].copy()
    if use.empty:
        raise RuntimeError('No unique section-intersection records found for monthly anomaly figure')
    use['month'] = pd.to_datetime(use['date']).dt.to_period('M').dt.to_timestamp()

    rows = []
    for s in SECTION_ORDER:
        q = use[use['section'] == s].copy()
        if q.empty:
            continue
        for product, col in [('ADT', 'ADT_radial_km'), ('GLORYS', 'GLORYS_radial_km')]:
            if col not in q.columns:
                raise KeyError(f'Missing column {col} in section table')
            monthly = q.groupby('month', as_index=False)[col].mean().rename(columns={col: 'position_km'})
            ref = monthly['position_km'].mean()
            monthly['product'] = product
            monthly['section'] = s
            monthly['anomaly_km'] = monthly['position_km'] - ref
            monthly['smooth5'] = monthly['anomaly_km'].rolling(5, min_periods=1, center=True).mean()
            rows.append(monthly)
    out = pd.concat(rows, ignore_index=True)
    return out


In [ ]:
# ==== 4. Figure 3: distance distribution / transect differences / long-term variations ====
def plot_figure3():
    daily, sec = load_r79_tables()
    daily = daily.dropna(subset=['D_mean_km']).sort_values('date').copy()
    annual = yearly_summary_from_daily(daily)

    fig, axs = plt.subplots(3, 1, figsize=(11.2, 14.8))

    # (a)
    ax = axs[0]
    v = daily['D_mean_km'].dropna().to_numpy(float)
    bins = np.arange(0, max(100, np.nanquantile(v, 0.995)*1.05), 1.5)
    ax.hist(v, bins=bins, density=True, color=COLORS['fill_blue'], edgecolor='white', alpha=.85)
    if len(v) > 10 and np.nanstd(v) > 1e-8:
        xs = np.linspace(0, 100, 450)
        density = gaussian_kde(v)(xs)
        ax.plot(xs, density, lw=2.8, color=COLORS['blue'])
    med = float(np.nanmedian(v))
    p95 = float(np.nanquantile(v, .95))
    ax.axvline(med, ls='--', lw=2.1, color=COLORS['orange'])
    ax.axvline(p95, ls='--', lw=2.1, color='#6B8E23')
    finish_axis(ax, xlabel='Daily symmetric mean path distance (km)', ylabel='Probability density', title='(a) Full-path distance distribution')
    ax.set_xlim(0, 100)
    box_text = f'Valid paired days: {len(v):,}\nMedian: {med:.1f} km\nP95: {p95:.1f} km'
    ax.text(0.985, 0.96, box_text, transform=ax.transAxes, ha='right', va='top', fontsize=LEGEND_SIZE,
            bbox=dict(boxstyle='round', facecolor='#F1F4F7', alpha=.9, edgecolor='#C7CED6'))

    # (b)
    ax = axs[1]
    samples = [sec.loc[sec['section'] == s, 'D_section_abs_km'].dropna().to_numpy(float) for s in SECTION_ORDER]
    boxes = []
    for x in samples:
        if len(x):
            boxes.append(dict(med=float(np.median(x)), q1=float(np.quantile(x, .25)), q3=float(np.quantile(x, .75)),
                              whislo=float(np.quantile(x, .05)), whishi=float(np.quantile(x, .95)), fliers=[]))
        else:
            boxes.append(dict(med=np.nan, q1=np.nan, q3=np.nan, whislo=np.nan, whishi=np.nan, fliers=[]))
    ax.bxp(boxes, positions=np.arange(1, 7), widths=.52, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor=COLORS['blue'], alpha=.18, edgecolor=COLORS['blue'], lw=1.6),
           medianprops=dict(color=COLORS['blue'], lw=2.3),
           whiskerprops=dict(color=COLORS['blue'], lw=1.2),
           capprops=dict(color=COLORS['blue'], lw=1.2))
    ax.set_xticks(range(1, 7), labels=SECTION_ORDER)
    finish_axis(ax, xlabel='Transect', ylabel='Absolute intersection-position difference (km)', title='(b) Transect position differences')

    # (c)
    ax = axs[2]
    g = daily.set_index('date')['D_mean_km'].sort_index()
    calendar = g.resample('D').mean()
    rolling = calendar.rolling('90D', min_periods=45).mean()
    ax.plot(calendar.index, calendar.values, color=COLORS['gray'], lw=.45, alpha=.18)
    ax.plot(rolling.index, rolling.values, color=COLORS['blue'], lw=2.0, alpha=.95)
    years = pd.to_datetime(annual['year'].astype(str) + '-07-01')
    ax.plot(years, annual['D_mean_km'].values, color=COLORS['orange'], marker='o', ms=4.5, lw=2.0)
    ax.fill_between(years, annual['P25'].values, annual['P75'].values, color=COLORS['orange'], alpha=.12)
    finish_axis(ax, xlabel='Year', ylabel='Daily symmetric mean path distance (km)', title='(c) Long-term distance variations')
    ax.set_xlim(calendar.index.min(), calendar.index.max())

    fig.subplots_adjust(hspace=.38)
    return save_figure(fig, 'Figure3_distance_distribution_transects_longterm_unified')


In [ ]:
# ==== 5. Figure 4: monthly transect-position anomalies ====
def plot_figure4():
    _, sec = load_r79_tables()
    mon = monthly_section_anomalies(sec)

    fig, axes = plt.subplots(3, 2, figsize=(14.8, 12.6), sharex=True)
    axes = axes.ravel()

    product_color = {'ADT': COLORS['blue'], 'GLORYS': COLORS['orange']}

    for ax, s in zip(axes, SECTION_ORDER):
        q = mon[mon['section'] == s].copy().sort_values('month')
        if q.empty:
            ax.set_visible(False)
            continue
        for prod in ['ADT', 'GLORYS']:
            p = q[q['product'] == prod]
            ax.plot(p['month'], p['anomaly_km'], color=product_color[prod], alpha=.45, lw=1.0)
            ax.plot(p['month'], p['smooth5'], color=product_color[prod], alpha=.95, lw=2.0, label=prod)
        ax.axhline(0, color='#B0B7BF', lw=1.0)
        finish_axis(ax, ylabel='Anomaly (km)', title=f'({chr(97+SECTION_ORDER.index(s))}) Section {s}')

    for ax in axes[-2:]:
        ax.set_xlabel('Year', fontsize=LABEL_SIZE)

    handles = [Line2D([0],[0], color=COLORS['blue'], lw=2, label='ADT'),
               Line2D([0],[0], color=COLORS['orange'], lw=2, label='GLORYS')]
    fig.legend(handles=handles, loc='upper center', ncol=2, frameon=False, fontsize=LEGEND_SIZE, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle('Monthly transect-position anomalies relative to each product mean', fontsize=TITLE_SIZE+2, fontweight='bold', y=0.995)
    fig.text(0.5, 0.02, 'Each reference mean is calculated from the same set of paired valid months; solid thick lines are display-only 5-month averages.',
             ha='center', fontsize=LEGEND_SIZE)
    fig.subplots_adjust(top=0.92, bottom=0.08, hspace=0.34, wspace=0.18)
    return save_figure(fig, 'Figure4_monthly_transect_position_anomalies_unified')


In [ ]:
# ==== 6. Figure 5: expanding-window rolling hindcasts ====
def locate_fig5_table():
    if FIG5_TABLE:
        return Path(FIG5_TABLE)
    return find_first(PROJECT_ROOT, ['**/08_逐年模型汇总.csv', '**/08_论文用各模型多年后报汇总.csv'])


def load_fig5_table():
    path = locate_fig5_table()
    df = read_csv_flex(path)
    print('[Figure5] table =', path)
    return df


def infer_primary_model(df):
    model_col = choose_column(df, ['模型', 'model'])
    lead_col = choose_column(df, ['提前期_天', 'lead_day', 'lead'])
    rmse_col = choose_column(df, ['横向误差RMSE_km', 'RMSE_km', 'rmse_km'])
    year_col = choose_column(df, ['测试年份', 'year', '年份'])

    if FIG5_MODEL_NAME is not None:
        return FIG5_MODEL_NAME
    models = sorted(df[model_col].dropna().astype(str).unique())
    preferred = [m for m in models if any(k in m.lower() for k in ['eof-tcn-lstm', 'hybrid', 'proposed', '主模型'])]
    if preferred:
        return preferred[0]
    tmp = df.groupby(model_col, as_index=False)[rmse_col].mean().sort_values(rmse_col)
    return str(tmp.iloc[0][model_col])


def plot_figure5():
    df = load_fig5_table()
    model_col = choose_column(df, ['模型', 'model'])
    lead_col = choose_column(df, ['提前期_天', 'lead_day', 'lead'])
    rmse_col = choose_column(df, ['横向误差RMSE_km', 'RMSE_km', 'rmse_km'])
    year_col = choose_column(df, ['测试年份', 'year', '年份'])

    main_model = infer_primary_model(df)
    q = df[df[model_col].astype(str) == str(main_model)].copy()
    if q.empty:
        raise RuntimeError(f'Figure5 main model not found: {main_model}')
    q[year_col] = q[year_col].astype(int)

    fig, ax = plt.subplots(figsize=(12.6, 7.2))
    color_map = {30: COLORS['blue'], 60: COLORS['orange'], 90: COLORS['green']}
    for lead in LEADS:
        p = q[q[lead_col].astype(int) == int(lead)].sort_values(year_col)
        if p.empty:
            continue
        ax.plot(p[year_col], p[rmse_col], marker='o', ms=8, lw=2.2, label=f'+{lead} days', color=color_map.get(lead))

    finish_axis(ax, xlabel='Hindcast test year', ylabel='Target-date path RMSE (km)',
                title='Hybrid deep-learning prediction | expanding-window rolling hindcasts')
    ax.legend(frameon=False, ncol=3, loc='upper left', fontsize=LEGEND_SIZE)
    return save_figure(fig, 'Figure5_expanding_window_rolling_hindcasts_unified')


In [ ]:
# ==== 7. Figure 6: auto-built comparison figure from summary / CI tables ====
def locate_fig6_table():
    if FIG6_TABLE:
        return Path(FIG6_TABLE)
    # 优先 R4 跨年份置信区间表，其次 R3 预测误差不确定性表
    candidates = [
        '**/09_预测误差_跨年份分块Bootstrap_95CI.csv',
        '**/prediction_uncertainty*.csv',
        '**/*95CI*.csv',
    ]
    return find_first(PROJECT_ROOT, candidates)


def plot_figure6():
    path = locate_fig6_table()
    df = read_csv_flex(path)
    print('[Figure6] table =', path)

    # 尝试适配长表结构
    model_col = choose_column(df, ['模型', 'model'])
    lead_col = choose_column(df, ['提前期_天', 'lead_day', 'lead'])
    mean_col = choose_column(df, ['横向误差RMSE均值_km', '横向误差RMSE_km', 'RMSE_km', 'rmse_km'])
    lo_col = choose_column(df, ['95CI下限_km', 'CI_low_km', 'ci_low_km', '下限_km'], required=False)
    hi_col = choose_column(df, ['95CI上限_km', 'CI_high_km', 'ci_high_km', '上限_km'], required=False)

    # 选前几个最重要模型
    model_order = list(pd.Series(df[model_col].dropna().astype(str).unique()).tolist())
    preferred_keys = ['Persistence', 'Climatology', 'AR-EOF', 'Matched-LSTM', 'Simple-LSTM', 'EOF-TCN-LSTM', 'EOF-TCN-GRU']
    ordered = [m for k in preferred_keys for m in model_order if m == k]
    for m in model_order:
        if m not in ordered:
            ordered.append(m)
    ordered = ordered[:6]

    fig, ax = plt.subplots(figsize=(11.0, 7.0))
    offsets = np.linspace(-1.3, 1.3, len(ordered))
    palette = plt.cm.tab10(np.linspace(0, 1, len(ordered)))

    for off, model, col in zip(offsets, ordered, palette):
        q = df[df[model_col].astype(str) == str(model)].copy()
        if q.empty:
            continue
        q = q.sort_values(lead_col)
        x = q[lead_col].astype(float).to_numpy() + off
        y = q[mean_col].astype(float).to_numpy()
        if lo_col and hi_col and lo_col in q.columns and hi_col in q.columns:
            lo = y - q[lo_col].astype(float).to_numpy()
            hi = q[hi_col].astype(float).to_numpy() - y
            ax.errorbar(x, y, yerr=np.vstack([lo, hi]), fmt='o-', capsize=3, color=col, label=model)
        else:
            ax.plot(x, y, 'o-', color=col, label=model)

    finish_axis(ax, xlabel='Forecast lead (days)', ylabel='Path RMSE (km)', title='Model comparison with unified confidence-interval styling')
    ax.set_xticks(LEADS)
    ax.legend(frameon=False, fontsize=LEGEND_SIZE, ncol=2)
    return save_figure(fig, 'Figure6_model_comparison_unified')


In [ ]:
# ==== 8. Figure 7: forecast path RMSE at six diagnostic transects ====
def locate_fig7_table():
    if FIG7_TABLE:
        return Path(FIG7_TABLE)
    return find_first(PROJECT_ROOT, FIG7_GLOB_PATTERNS)


def load_fig7_table():
    path = locate_fig7_table()
    df = read_csv_flex(path)
    print('[Figure7] table =', path)
    return df


def plot_figure7():
    df = load_fig7_table()
    sec_col = choose_column(df, ['section', 'transect', '断面', 'section_name'])
    lead_col = choose_column(df, ['lead_day', 'lead', '提前期_天'])
    model_col = choose_column(df, ['model', '模型'])
    rmse_col = choose_column(df, ['rmse_km', 'RMSE_km', '横向误差RMSE_km', 'path_rmse_km'])

    sections_present = [s for s in SECTION_ORDER if s in set(df[sec_col].astype(str))]
    if not sections_present:
        raise RuntimeError('Figure7 table does not contain recognizable section labels A/B/PN/C/D/E')

    # 选择最多 5 条模型曲线，避免过拥挤
    model_order = list(pd.Series(df[model_col].dropna().astype(str).unique()).tolist())
    preferred = ['Persistence', 'Climatology', 'AR-EOF', 'Matched-LSTM', 'Simple-LSTM', 'EOF-TCN-LSTM', 'EOF-TCN-GRU', 'Hybrid']
    models = [m for k in preferred for m in model_order if m == k]
    for m in model_order:
        if m not in models:
            models.append(m)
    models = models[:5]

    fig, axes = plt.subplots(3, 2, figsize=(13.8, 16.5), sharex=True)
    axes = axes.ravel()
    palette = plt.cm.tab10(np.linspace(0, 1, len(models)))

    for i, s in enumerate(SECTION_ORDER):
        ax = axes[i]
        q0 = df[df[sec_col].astype(str) == s].copy()
        if q0.empty:
            ax.set_visible(False)
            continue
        for model, col in zip(models, palette):
            q = q0[q0[model_col].astype(str) == str(model)].copy().sort_values(lead_col)
            if q.empty:
                continue
            ax.plot(q[lead_col].astype(float), q[rmse_col].astype(float), marker='o', ms=4.5, lw=1.8, color=col, label=model)
        finish_axis(ax, xlabel='Forecast lead (days)' if i >= 4 else None, ylabel='RMSE (km)', title=f'({chr(97+i)}) Section {s}')
        ax.set_xticks(LEADS)

    handles = [Line2D([0],[0], color=c, lw=2, label=m) for m, c in zip(models, palette)]
    fig.legend(handles=handles, loc='upper center', ncol=min(3, len(models)), frameon=False, fontsize=LEGEND_SIZE, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle('Forecast path RMSE at six diagnostic transects', fontsize=TITLE_SIZE+2, fontweight='bold', y=0.995)
    fig.subplots_adjust(top=0.94, hspace=0.36, wspace=0.22)
    return save_figure(fig, 'Figure7_six_diagnostic_transects_rmse_unified')


In [ ]:
# ==== 9. Figure 8: input-noise sensitivity + random-seed stability ====
def locate_fig8_noise_table():
    if FIG8_NOISE_TABLE:
        return Path(FIG8_NOISE_TABLE)
    return find_first(PROJECT_ROOT, FIG8_NOISE_GLOB_PATTERNS)


def locate_fig8_seed_table():
    if FIG8_SEED_TABLE:
        return Path(FIG8_SEED_TABLE)
    return find_first(PROJECT_ROOT, FIG8_SEED_GLOB_PATTERNS)


def plot_figure8():
    noise_path = locate_fig8_noise_table()
    seed_path = locate_fig8_seed_table()
    noise = read_csv_flex(noise_path)
    seed = read_csv_flex(seed_path)
    print('[Figure8] noise table =', noise_path)
    print('[Figure8] seed  table =', seed_path)

    n_lead = choose_column(noise, ['lead_day', 'lead', '提前期_天'])
    n_sd = choose_column(noise, ['noise_sd_km', 'input_noise_sd_km', 'sd_km', '输入噪声标准差_km'])
    n_delta = choose_column(noise, ['delta_rmse_km', 'change_rmse_km', 'rmse_change_km', '相对clean输入的RMSE变化_km'])
    n_lo = choose_column(noise, ['ci_low_km', 'CI_low_km', '下限_km'], required=False)
    n_hi = choose_column(noise, ['ci_high_km', 'CI_high_km', '上限_km'], required=False)

    s_lead = choose_column(seed, ['lead_day', 'lead', '提前期_天'])
    s_year = choose_column(seed, ['year', '测试年份', '年份'])
    s_sd = choose_column(seed, ['rmse_sd_km', 'seed_sd_km', 'Across-seed RMSE SD (km)', '跨随机种子RMSE标准差_km'])

    fig, axes = plt.subplots(1, 2, figsize=(15.8, 6.3))

    # (a)
    ax = axes[0]
    cmap = {30: COLORS['blue'], 60: COLORS['orange'], 90: COLORS['green']}
    for lead in LEADS:
        q = noise[noise[n_lead].astype(int) == int(lead)].copy().sort_values(n_sd)
        if q.empty:
            continue
        ax.plot(q[n_sd].astype(float), q[n_delta].astype(float), marker='o', ms=5.5, lw=2.0, color=cmap.get(lead), label=f'+{lead} days')
        if n_lo and n_hi and n_lo in q.columns and n_hi in q.columns:
            y = q[n_delta].astype(float).to_numpy()
            lo = y - q[n_lo].astype(float).to_numpy()
            hi = q[n_hi].astype(float).to_numpy() - y
            ax.errorbar(q[n_sd].astype(float), y, yerr=np.vstack([lo, hi]), fmt='none', color=cmap.get(lead), alpha=.45, capsize=3)
    ax.axhline(0, color='#6B6B6B', ls='--', lw=1.4)
    finish_axis(ax, xlabel='Input Gaussian position-noise SD (km)', ylabel='Change in RMSE relative to clean input (km)', title='(a) Input noise sensitivity')
    ax.legend(frameon=False, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.18), fontsize=LEGEND_SIZE)

    # (b)
    ax = axes[1]
    for lead in LEADS:
        q = seed[seed[s_lead].astype(int) == int(lead)].copy().sort_values(s_year)
        if q.empty:
            continue
        ax.plot(q[s_year].astype(int), q[s_sd].astype(float), marker='o', ms=5.5, lw=2.0, color=cmap.get(lead), label=f'+{lead} days')
    finish_axis(ax, xlabel='Hindcast test year', ylabel='Across-seed RMSE SD (km)', title='(b) Random-seed stability')

    fig.subplots_adjust(wspace=0.28, bottom=0.20)
    return save_figure(fig, 'Figure8_noise_sensitivity_and_seed_stability_unified')


In [ ]:
# ==== 10. Run individual figures ====
# 按需逐个运行。建议先从 Figure 2~6 开始。


In [ ]:
# Example: run one figure
# plot_figure2()
# plot_figure3()
# plot_figure4()
# plot_figure5()
# plot_figure6()
# plot_figure7()
# plot_figure8()


In [ ]:
# ==== 11. Run all available figures ====
results = {}
for name, func in [
    ('Figure2', plot_figure2),
    ('Figure3', plot_figure3),
    ('Figure4', plot_figure4),
    ('Figure5', plot_figure5),
    ('Figure6', plot_figure6),
    ('Figure7', plot_figure7),
    ('Figure8', plot_figure8),
]:
    try:
        results[name] = func()
    except Exception as e:
        results[name] = f'FAILED: {e}'
        print(f'[WARN] {name} failed -> {e}')

results
